# MIRACLE — STEW Model Development: Baseline, Normalization, Feature Engineering & Model Comparison

Full pipeline: load extracted STEW features, evaluate with subject-level LOSO,
test normalization strategies, engineer new features (alpha/beta ratio, frontal
asymmetry), compare multiple models, and tune hyperparameters to find the best
overall setup for the channel-ablation step.

## 1. Load features & LOSO evaluation helper

In [1]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from xgboost import XGBClassifier

data = np.load('stew_features_two_class.npz', allow_pickle=True)
X, y, groups = data['X'], data['y'], data['groups']
channels = list(data['channels'])
bands = list(data['bands'])

print(f"X shape: {X.shape}")
print(f"Subjects: {len(np.unique(groups))}")
print(f"Class balance: {dict(zip(*np.unique(y, return_counts=True)))}")

X shape: (6705, 70)
Subjects: 45
Class balance: {np.int64(0): np.int64(2980), np.int64(1): np.int64(3725)}


**Important note on the metric:** labels are subject-level (one rating per subject
covers all their epochs), so a held-out subject's test set is single-class. Raw
per-fold epoch accuracy is noisy and not meaningful on its own.

The right metric: take a **majority vote across each held-out subject's epoch
predictions** → one prediction per subject → score across all 45 subjects.

In [2]:
def run_loso(clf_factory, X, y, groups, scale=False):
    """clf_factory: zero-arg function returning a fresh, unfitted classifier."""
    logo = LeaveOneGroupOut()
    subject_true, subject_pred = [], []
    for train_idx, test_idx in logo.split(X, y, groups):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train = y[train_idx]
        if scale:
            mu = X_train.mean(axis=0, keepdims=True)
            sigma = X_train.std(axis=0, keepdims=True) + 1e-8
            X_train = (X_train - mu) / sigma
            X_test = (X_test - mu) / sigma
        clf = clf_factory()
        clf.fit(X_train, y_train)
        pred_epochs = clf.predict(X_test)
        majority = np.bincount(pred_epochs.astype(int)).argmax()
        subject_true.append(y[test_idx][0])
        subject_pred.append(majority)
    subject_true, subject_pred = np.array(subject_true), np.array(subject_pred)
    acc = accuracy_score(subject_true, subject_pred)
    bal_acc = balanced_accuracy_score(subject_true, subject_pred)
    n_correct = (subject_true == subject_pred).sum()
    return acc, bal_acc, n_correct

## 2. Baseline: Random Forest, all 14 channels, absolute band power

In [3]:
acc, bal_acc, n_correct = run_loso(
    lambda: RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1),
    X, y, groups
)
print(f"Baseline RF — accuracy: {acc:.3f} ({n_correct}/45), balanced: {bal_acc:.3f}")

Baseline RF — accuracy: 0.578 (26/45), balanced: 0.550


## 3. Does normalization help?

Absolute band power varies naturally person-to-person (skull thickness, electrode
contact). Tested two normalization strategies to see if removing that helps
generalization to a new subject.

In [4]:
n_channels, n_bands = 14, 5
X_reshaped = X.reshape(X.shape[0], n_channels, n_bands)
power = np.expm1(X_reshaped)  # undo log1p to get raw power

# Relative power: each channel's band power as fraction of that channel's total power
total_power = power.sum(axis=-1, keepdims=True) + 1e-10
X_relative = np.log1p(power / total_power).reshape(X.shape[0], -1)

acc_rel, bal_rel, n_rel = run_loso(
    lambda: RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1),
    X_relative, y, groups
)
print(f"Relative power — accuracy: {acc_rel:.3f} ({n_rel}/45), balanced: {bal_rel:.3f}")

Relative power — accuracy: 0.356 (16/45), balanced: 0.335


In [5]:
# Per-subject z-score: standardize each feature using that subject's own epoch mean/std
X_z = np.zeros_like(X)
for subj in np.unique(groups):
    mask = groups == subj
    mu = X[mask].mean(axis=0, keepdims=True)
    sigma = X[mask].std(axis=0, keepdims=True) + 1e-8
    X_z[mask] = (X[mask] - mu) / sigma

acc_z, bal_z, n_z = run_loso(
    lambda: RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1),
    X_z, y, groups
)
print(f"Per-subject z-score — accuracy: {acc_z:.3f} ({n_z}/45), balanced: {bal_z:.3f}")

Per-subject z-score — accuracy: 0.422 (19/45), balanced: 0.380


**Both normalizations made things worse**, not better. This makes sense: since the
label is subject-level (a subject is entirely "normal" or entirely "high"), the
between-subject baseline offset in absolute power IS part of the signal we're
trying to classify. Per-subject normalization removes exactly that signal.

→ **Sticking with absolute band power going forward.**

## 4. Feature engineering: alpha/beta ratio + frontal asymmetry

Two additional feature groups, both known markers of cognitive workload in the
EEG literature:
- **Alpha/beta ratio per channel** — classic workload indicator (ratio drops as workload rises)
- **Frontal asymmetry** — log(right) − log(left) alpha power for 4 frontal channel pairs

In [6]:
alpha_idx, beta_idx = bands.index('alpha'), bands.index('beta')
alpha_power = power[:, :, alpha_idx]
beta_power = power[:, :, beta_idx]

# Alpha/beta ratio per channel
ab_ratio_log = np.log1p(alpha_power / (beta_power + 1e-8))

# Frontal asymmetry (log-right minus log-left alpha power)
frontal_pairs = [("F3", "F4"), ("F7", "F8"), ("FC5", "FC6"), ("AF3", "AF4")]
asymmetry_feats = []
for left, right in frontal_pairs:
    li, ri = channels.index(left), channels.index(right)
    asymmetry_feats.append(np.log1p(alpha_power[:, ri]) - np.log1p(alpha_power[:, li]))
asymmetry_feats = np.stack(asymmetry_feats, axis=-1)

# Combine: original 70 + ratio (14) + asymmetry (4) = 88 features
X_extended = np.concatenate([X, ab_ratio_log, asymmetry_feats], axis=-1)
print(f"Original features: {X.shape[1]}  ->  Extended features: {X_extended.shape[1]}")

Original features: 70  ->  Extended features: 88


## 5. Model comparison (baseline vs. extended features)

Testing Logistic Regression, SVM (RBF), XGBoost, and Random Forest — all with
subject-level LOSO.

In [7]:
acc, bal, n = run_loso(lambda: LogisticRegression(max_iter=1000), X, y, groups, scale=True)
print(f"LogReg (baseline, 70 feat):  acc={acc:.3f} ({n}/45), bal={bal:.3f}")
acc, bal, n = run_loso(lambda: LogisticRegression(max_iter=1000), X_extended, y, groups, scale=True)
print(f"LogReg (extended, 88 feat): acc={acc:.3f} ({n}/45), bal={bal:.3f}")

LogReg (baseline, 70 feat):  acc=0.444 (20/45), bal=0.425
LogReg (extended, 88 feat): acc=0.444 (20/45), bal=0.425


In [8]:
acc, bal, n = run_loso(lambda: SVC(kernel='rbf', C=1.0), X, y, groups, scale=True)
print(f"SVM-rbf (baseline):  acc={acc:.3f} ({n}/45), bal={bal:.3f}")
acc, bal, n = run_loso(lambda: SVC(kernel='rbf', C=1.0), X_extended, y, groups, scale=True)
print(f"SVM-rbf (extended):  acc={acc:.3f} ({n}/45), bal={bal:.3f}")

SVM-rbf (baseline):  acc=0.444 (20/45), bal=0.425
SVM-rbf (extended):  acc=0.422 (19/45), bal=0.405


In [9]:
def make_xgb():
    return XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                          eval_metric='logloss', verbosity=0, n_jobs=1)

acc, bal, n = run_loso(make_xgb, X, y, groups)
print(f"XGBoost (baseline):  acc={acc:.3f} ({n}/45), bal={bal:.3f}")
acc, bal, n = run_loso(make_xgb, X_extended, y, groups)
print(f"XGBoost (extended):  acc={acc:.3f} ({n}/45), bal={bal:.3f}")

XGBoost (baseline):  acc=0.489 (22/45), bal=0.465
XGBoost (extended):  acc=0.511 (23/45), bal=0.485


In [10]:
acc, bal, n = run_loso(
    lambda: RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1),
    X_extended, y, groups
)
print(f"RF default (extended):  acc={acc:.3f} ({n}/45), bal={bal:.3f}")

RF default (extended):  acc=0.511 (23/45), bal=0.485


## 6. Hyperparameter tuning — is Random Forest overfitting?

With only 45 subjects and 70-88 features, plain Random Forest (unlimited depth)
may be overfitting to individual subjects' noise. Testing a depth-limited,
more conservative RF.

In [11]:
def make_rf_tuned():
    return RandomForestClassifier(
        n_estimators=200, max_depth=6, min_samples_leaf=5,
        random_state=42, n_jobs=1
    )

acc_base_tuned, bal_base_tuned, n_base_tuned = run_loso(make_rf_tuned, X, y, groups)
print(f"RF tuned, baseline (70 feat):  acc={acc_base_tuned:.3f} ({n_base_tuned}/45), bal={bal_base_tuned:.3f}")

acc_ext_tuned, bal_ext_tuned, n_ext_tuned = run_loso(make_rf_tuned, X_extended, y, groups)
print(f"RF tuned, extended (88 feat): acc={acc_ext_tuned:.3f} ({n_ext_tuned}/45), bal={bal_ext_tuned:.3f}")

RF tuned, baseline (70 feat):  acc=0.622 (28/45), bal=0.580
RF tuned, extended (88 feat): acc=0.644 (29/45), bal=0.600


## 7. Summary & conclusion

| Model | Features | Subject accuracy | Balanced accuracy |
|---|---|---|---|
| Random Forest (default) | absolute power (70) | 0.578 | 0.550 |
| Random Forest (default) | relative power (70) | 0.356 | 0.335 |
| Random Forest (default) | per-subject z-score (70) | 0.422 | 0.380 |
| Logistic Regression | baseline (70) | 0.444 | 0.425 |
| Logistic Regression | extended (88) | 0.444 | 0.425 |
| SVM (RBF) | baseline (70) | 0.444 | 0.425 |
| SVM (RBF) | extended (88) | 0.422 | 0.405 |
| XGBoost | baseline (70) | 0.489 | 0.465 |
| XGBoost | extended (88) | 0.511 | 0.485 |
| Random Forest (default) | extended (88) | 0.511 | 0.485 |
| **Random Forest (tuned, depth=6)** | **baseline (70)** | **0.622** | **0.580** |
| **Random Forest (tuned, depth=6)** | **extended (88)** | **0.644** | **0.600** |

**Best setup: Random Forest with limited depth (`max_depth=6, min_samples_leaf=5`)
on the extended feature set (absolute band power + alpha/beta ratio + frontal
asymmetry) — 64.4% subject-level accuracy (29/45), balanced accuracy 0.600.**

Key takeaways:
1. Absolute band power carries real between-subject signal — normalizing it away hurts.
2. The default (unconstrained) Random Forest was overfitting on this small sample (45 subjects); limiting tree depth alone improved accuracy by +4.4 points.
3. Adding alpha/beta ratio and frontal asymmetry features gave a further +2.2 points.
4. Simpler models (logistic regression, SVM) and XGBoost all underperformed the tuned Random Forest here — likely because RF's flexibility (once regularized) fits this feature set well without over-relying on any single dimension.

**Next step: use this tuned RF + extended-feature setup as the basis for the channel ablation experiment (14 → Muse montage → 2 → 1 channels).**